In [1]:
from iskna_ecg_xgboost_v1 import make_fold_id, mantis_xgb, ecgfounder_xgb
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from tabpfn import TabPFNClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from natsort import natsorted
from tqdm import tqdm
import gc
import sys
sys.path.append(r'V:\dunwei\miSKNA\program\tools')
from tools import METRICS
gc.collect()
torch.cuda.empty_cache()

In [2]:
def tabpfn_feature_process(iskna_label_dict, iskna_proba_dict, ecgfounder_label_dict, ecgfounder_proba_dict, dataset):
    if set(iskna_label_dict['fold_1']['research_id']) != set(ecgfounder_label_dict['fold_1']['research_id']):
        raise ValueError("iskna_label_ID and ecgfounder_label_ID do not match.")
    fold_keys = iskna_label_dict.keys()
    tmp_label_list = []
    tmp_iskna_proba_list = []
    tmp_ecgfounder_proba_list = []
    for fold in fold_keys:
        if set(iskna_label_dict[fold][f'y_{dataset}']) != set(ecgfounder_label_dict[fold][f'y_{dataset}']):
            raise ValueError(f"iskna_label_dict and ecgfounder_label_dict do not match in fold {fold}.")
        
        tmp_label_list.append(iskna_label_dict[fold][['research_id', f'y_{dataset}']])
        tmp_iskna_proba_list.append(iskna_proba_dict[fold][['research_id', f'yhat_{dataset}_proba']])
        tmp_ecgfounder_proba_list.append(ecgfounder_proba_dict[fold][['research_id', f'yhat_{dataset}_proba']])

    label_concat_df = pd.concat(tmp_label_list, ignore_index=True)
    iskna_proba_concat_df = pd.concat(tmp_iskna_proba_list, ignore_index=True)
    ecgfounder_proba_concat_df = pd.concat(tmp_ecgfounder_proba_list, ignore_index=True)

    y_subject = label_concat_df.groupby('research_id')[f'y_{dataset}'].first().reset_index()
    yhat_iskna_proba = iskna_proba_concat_df.groupby('research_id')[f'yhat_{dataset}_proba'].mean().reset_index()
    yhat_iskna_proba = yhat_iskna_proba.rename(columns={f'yhat_{dataset}_proba': f'iskna_yhat_{dataset}_proba'})
    yhat_ecgfounder_proba = ecgfounder_proba_concat_df.groupby('research_id')[f'yhat_{dataset}_proba'].mean().reset_index()
    yhat_ecgfounder_proba = yhat_ecgfounder_proba.rename(columns={f'yhat_{dataset}_proba': f'ecgfounder_yhat_{dataset}_proba'})


    merged_data_df = pd.merge(y_subject, yhat_iskna_proba, on='research_id')
    merged_data_df = pd.merge(merged_data_df, yhat_ecgfounder_proba, on='research_id')

    
    return merged_data_df

In [3]:
iskna_dataset_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\iskna_ecg_xgboost\sr10k_500-3.5k_miskna_2-7min_trH126P126win_teH126P126win_512mantis_split\dataset/'
iskna_save_data_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\iskna_ecg_xgboost\sr10k_500-3.5k_miskna_2-7min_trH126P126win_teH126P126win_512mantis_split\scaler_0.95pca/'

ecgfounder_dataset_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\iskna_ecg_xgboost\sr500_0.5-50_ecg_2-7min_10win2stride_trH146P146win_teH146P146win_1024ecgfounder_split\dataset/'
ecgfounder_save_data_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\iskna_ecg_xgboost\sr500_0.5-50_ecg_2-7min_10win2stride_trH146P146win_teH146P146win_1024ecgfounder_split/'

tabpfn_save_feature_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\xgboost_tabpfn\dataset/'
tabpfn_save_data_path = r'V:\dunwei\miSKNA_MACE\dataset\meta_learner_result\v5\xgboost_tabpfn/'

In [4]:
TRAIN_IDS = pd.read_csv(r"V:\dunwei\miSKNA_MACE\dataset\MI_MACE_SKNA\train_IDs.csv", dtype={'research_id': str})
TEST_IDS = pd.read_csv(r"V:\dunwei\miSKNA_MACE\dataset\MI_MACE_SKNA\test_IDs.csv", dtype={'research_id': str})

hyper_params_dict = {
    'objective': 'binary:logistic','booster': 'gbtree', 'eval_metric': 'aucpr', 'learning_rate': 0.05, 'n_estimators': 500, 'max_depth': 3, 'min_child_weight': 1,
    'gamma': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.01, 'reg_lambda': 1, 'random_state': 42
}
hyper_params_df = pd.DataFrame({'hyper_param': list(hyper_params_dict.keys()), 'value': list(hyper_params_dict.values())})
hyper_params_df.to_csv(os.path.join(iskna_dataset_path, 'hyper_params.csv'), index=False)
hyper_params_df.to_csv(os.path.join(ecgfounder_dataset_path, 'hyper_params.csv'), index=False)

In [ ]:
runs = 100

for run in range(runs):
    iskna_save_run_path = os.path.join(iskna_save_data_path, f'run_{run + 1}/')
    os.makedirs(iskna_save_run_path, exist_ok=True)
    ecgfounder_save_run_path = os.path.join(ecgfounder_save_data_path, f'run_{run + 1}/')
    os.makedirs(ecgfounder_save_run_path, exist_ok=True)
    tabpfn_save_run_path = os.path.join(tabpfn_save_data_path, f'run_{run + 1}/')
    os.makedirs(tabpfn_save_run_path, exist_ok=True)

    print(f'\n========== Run {run + 1} / {runs} ==========')

    fold_id_splits = make_fold_id(TRAIN_IDS, n_splits=10, random_state=42+run)

    print('\n====== MANTIS ======')
    train_iskna_label_dict, train_iskna_proba_dict, valid_iskna_label_dict, valid_iskna_proba_dict, test_iskna_label_dict, test_iskna_proba_dict = mantis_xgb(iskna_dataset_path, iskna_save_run_path, TRAIN_IDS, TEST_IDS, fold_id_splits, hyper_params_dict)
    print('\n====== ECG ======')
    train_ecgfounder_label_dict, train_ecgfounder_proba_dict, valid_ecgfounder_label_dict, valid_ecgfounder_proba_dict, test_ecgfounder_label_dict, test_ecgfounder_proba_dict = ecgfounder_xgb(ecgfounder_dataset_path, ecgfounder_save_run_path, TRAIN_IDS, TEST_IDS, fold_id_splits, hyper_params_dict)
    print('\n====== TabPFN  ======')
    # train_label_feature = tabpfn_feature_process(train_iskna_label_dict, train_ecgfounder_label_dict, train_iskna_proba_dict, train_ecgfounder_proba_dict, dataset='train')
    valid_label_feature_df = tabpfn_feature_process(valid_iskna_label_dict, valid_iskna_proba_dict, valid_ecgfounder_label_dict, valid_ecgfounder_proba_dict, dataset='valid')
    test_label_feature_df = tabpfn_feature_process(test_iskna_label_dict, test_iskna_proba_dict, test_ecgfounder_label_dict, test_ecgfounder_proba_dict, dataset='test')

    del train_iskna_label_dict, train_iskna_proba_dict
    del valid_iskna_label_dict, valid_iskna_proba_dict
    del test_iskna_label_dict, test_iskna_proba_dict

    del train_ecgfounder_label_dict, train_ecgfounder_proba_dict
    del valid_ecgfounder_label_dict, valid_ecgfounder_proba_dict
    del test_ecgfounder_label_dict, test_ecgfounder_proba_dict

    gc.collect()
    torch.cuda.empty_cache()

    tabpfn_save_feats_path = os.path.join(tabpfn_save_feature_path, f'run_{run + 1}/')
    os.makedirs(tabpfn_save_feats_path, exist_ok=True)
    valid_label_feature_df.to_csv(os.path.join(tabpfn_save_feats_path, f'meta_train_label_feature.csv'), index=False)
    test_label_feature_df.to_csv(os.path.join(tabpfn_save_feats_path, f'meta_test_label_feature.csv'), index=False)

    meta_test = test_label_feature_df[['y_test', 'iskna_yhat_test_proba', 'ecgfounder_yhat_test_proba']].to_numpy()
    meta_test_ids = test_label_feature_df['research_id'].tolist()

    X_meta_test, y_meta_test = meta_test[:, 1:], meta_test[:, 0]


    for fold, (train_group_ids, valid_group_ids) in enumerate(fold_id_splits):
        print(f'Fold {fold + 1} / {len(fold_id_splits)}')

        meta_train_list, meta_train_ids = [], []
        meta_valid_list, meta_valid_ids = [], []

        for train_id in train_group_ids:
            mask = valid_label_feature_df['research_id'].isin([train_id])
            train_data = valid_label_feature_df[mask]
            meta_train_list.append(train_data)
            meta_train_ids.extend([str(train_id)] * train_data.shape[0])

        for valid_id in valid_group_ids:
            mask = valid_label_feature_df['research_id'].isin([valid_id])
            valid_data = valid_label_feature_df[mask]
            meta_valid_list.append(valid_data)
            meta_valid_ids.extend([str(valid_id)] * valid_data.shape[0])


        meta_train_concat = pd.concat(meta_train_list, ignore_index=True)
        meta_train = meta_train_concat[['y_valid', 'iskna_yhat_valid_proba', 'ecgfounder_yhat_valid_proba']].to_numpy()
        meta_valid_concat = pd.concat(meta_valid_list, ignore_index=True)
        meta_valid = meta_valid_concat[['y_valid', 'iskna_yhat_valid_proba', 'ecgfounder_yhat_valid_proba']].to_numpy()

        X_meta_train, y_meta_train = meta_train[:, 1:], meta_train[:, 0]
        X_meta_valid, y_meta_valid = meta_valid[:, 1:], meta_valid[:, 0]
        print(f'Meta-train shape: {X_meta_train.shape}, Meta-valid shape: {X_meta_valid.shape}, Meta-test shape: {X_meta_test.shape}')
        
        clf = TabPFNClassifier(model_path="auto", random_state=42, device='cuda')
        clf.fit(X_meta_train, y_meta_train)

        yhat_train_proba = clf.predict_proba(X_meta_train)[:, 1]
        yhat_train = (yhat_train_proba >= 0.5).astype(int)

        yhat_valid_proba = clf.predict_proba(X_meta_valid)[:, 1]
        yhat_valid = (yhat_valid_proba >= 0.5).astype(int)

        yhat_test_proba = clf.predict_proba(X_meta_test)[:, 1]
        yhat_test = (yhat_test_proba >= 0.5).astype(int)

        y_train_label = pd.DataFrame({'research_id': meta_train_ids, 'y_train': y_meta_train, 'yhat_train': yhat_train})
        yhat_train_probability = pd.DataFrame({'research_id': meta_train_ids, 'yhat_train_proba': yhat_train_proba})
        y_valid_label = pd.DataFrame({'research_id': meta_valid_ids, 'y_valid': y_meta_valid, 'yhat_valid': yhat_valid})
        yhat_valid_probability = pd.DataFrame({'research_id': meta_valid_ids, 'yhat_valid_proba': yhat_valid_proba})
        y_test_label = pd.DataFrame({'research_id': meta_test_ids, 'y_test': y_meta_test, 'yhat_test': yhat_test})
        yhat_test_probability = pd.DataFrame({'research_id': meta_test_ids, 'yhat_test_proba': yhat_test_proba})

        y_train_label.to_csv(os.path.join(tabpfn_save_run_path, f'y_train_label_{fold+1}.csv'), index=False)
        yhat_train_probability.to_csv(os.path.join(tabpfn_save_run_path, f'yhat_train_probability_{fold+1}.csv'), index=False)
        y_valid_label.to_csv(os.path.join(tabpfn_save_run_path, f'y_valid_label_{fold+1}.csv'), index=False)
        yhat_valid_probability.to_csv(os.path.join(tabpfn_save_run_path, f'yhat_valid_probability_{fold+1}.csv'), index=False)
        y_test_label.to_csv(os.path.join(tabpfn_save_run_path, f'y_test_label_{fold+1}.csv'), index=False)
        yhat_test_probability.to_csv(os.path.join(tabpfn_save_run_path, f'yhat_test_probability_{fold+1}.csv'), index=False)

        del clf
        del meta_train, meta_train_concat, meta_valid, meta_valid_concat
        del X_meta_train, X_meta_valid
        del y_meta_train, y_meta_valid
        del yhat_train, yhat_valid, yhat_test
        del yhat_train_proba, yhat_valid_proba, yhat_test_proba
        del y_train_label, y_valid_label, y_test_label
        del yhat_train_probability, yhat_valid_probability, yhat_test_probability
        del meta_train_list, meta_train_ids
        del meta_valid_list, meta_valid_ids

        gc.collect()
        torch.cuda.empty_cache()

    del valid_label_feature_df
    del test_label_feature_df
    del meta_test, meta_test_ids
    del X_meta_test, y_meta_test
    del fold_id_splits
 
    gc.collect()
    torch.cuda.empty_cache()
        


========== Run 1 / 100 ==========

====== MANTIS ======
========== Processing Layer 1 / 6 ==========
--- Fold 1 / 10 ---
Train shape: (52164, 512), Valid shape: (5796, 512), Test shape: (68796, 512)
PCA X_Train shape : (52164, 23), PCA X_Valid shape : (5796, 23), PCA X_Test shape : (68796, 23)
--- Fold 2 / 10 ---
Train shape: (52164, 512), Valid shape: (5796, 512), Test shape: (68796, 512)
PCA X_Train shape : (52164, 23), PCA X_Valid shape : (5796, 23), PCA X_Test shape : (68796, 23)
--- Fold 3 / 10 ---
Train shape: (52164, 512), Valid shape: (5796, 512), Test shape: (68796, 512)
PCA X_Train shape : (52164, 23), PCA X_Valid shape : (5796, 23), PCA X_Test shape : (68796, 23)
--- Fold 4 / 10 ---
Train shape: (52164, 512), Valid shape: (5796, 512), Test shape: (68796, 512)
PCA X_Train shape : (52164, 23), PCA X_Valid shape : (5796, 23), PCA X_Test shape : (68796, 23)
--- Fold 5 / 10 ---
Train shape: (52164, 512), Valid shape: (5796, 512), Test shape: (68796, 512)
PCA X_Train shape : (521